###Spark SQL Advanced Functions

###Aggrigate functions

In [0]:
%sql
select array_agg(col) as array_col
 from (values (1), (2), (3), (4)) as tab (col)

In [0]:
df_csv=spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")


display(df_csv)

In [0]:
df_csv.createOrReplaceTempView("sql_tbl")

In [0]:
%sql
select customer_id,
    array_agg(order_id)
from sql_tbl
group by customer_id;

In [0]:
from pyspark.sql.functions import array_agg


display(df_csv.groupBy("customer_id").agg(array_agg("order_id")))

####Collect List
####Collect Set

In [0]:
%sql
select customer_id,
    collect_list(product_id),
    collect_set(product_id)
from sql_tbl
group by customer_id;
|

In [0]:
%sql
select
    corr(price, quantity),
    max(price),
    min(price),
    avg(price),
    median(price),
    mode(price)
from sql_tbl;

###Struct & Map

In [0]:
%sql
select struct(1,2,3,"abc")

In [0]:
%sql
select named_struct("a",1,"b",2,"c",3)

In [0]:
%sql
select map("a",1,"b",2,"c",3)

In [0]:
%sql
select map_values(map("a",1,"b",2,"c",3))

###DateTime Funcations

In [0]:
%sql
select current_timestamp();

In [0]:
%sql
select add_months(current_timestamp(),1);



In [0]:
%sql
select add_months(current_timestamp(),-1);

In [0]:
%sql
select convert_timezone('America/Los_Angeles',current_timestamp())

####Unix Time

In [0]:
%sql
select unix_timestamp()

In [0]:
%sql
select date_from_unix_date(1);

###Days

In [0]:
%sql
select dayname(current_timestamp()) as day,
    dayofmonth(current_timestamp()) as date,
    dayofweek(current_timestamp()) as weekday,
    dayofyear(current_timestamp()) as dayofyear

###Extract

In [0]:
%sql
select
    extract(year from current_timestamp()),
    extract(month from current_timestamp()),
    extract(day from current_timestamp()),
    extract(hour from current_timestamp()),
    extract(minute from current_timestamp()),
    extract(second from current_timestamp()),
    extract(doy from current_timestamp())

In [0]:
%sql
select from_unixtime()

In [0]:
%sql
select from_utc_timestamp(current_timestamp(),'America/Los_Angeles')

In [0]:
%sql
select from_unixtime(unix_timestamp(current_timestamp()))

###Window Functions

In [0]:
%sql
select * from sql_tbl;

In [0]:
%sql
select
    order_id,
    order_date,
    lag(order_date, 1, '1900-01-01') over (order by order_id) as prev_order_date,
    lead(order_date, 1, '2099-12-31') over (order by order_id) as next_order_date
from sql_tbl
limit 10

In [0]:
%sql
select 
    order_id,
    customer_id,
    order_status,
    ntile(3) over (partition by order_status order by order_id) as order_group
from sql_tbl;

###Array functions

In [0]:
%sql
select array(1, 2, 3)

In [0]:
%sql
select array_append(array(1,2,3),4)

In [0]:
%sql
select array_distinct(array(1,2,3,3))

In [0]:
%sql
select array_compact(array(1,2,NULL,3,NULL,3));

In [0]:
%sql
select array_contains(array(1,2,3),2)

In [0]:
%sql
select array_insert(array(1,2,3),2,9)

In [0]:
%sql
select array_prepend(array(1,2,3),0)

In [0]:
%sql
select array_position(array(1,2,3),2);

###UDF's In SparkSQL

In [0]:
def my_upper_function(input:str)->str:
    return input.upper()

In [0]:
my_upper_function('abc')

In [0]:
from pyspark.sql.types import StringType

spark.udf.register("my_upper", my_upper_function, StringType())

In [0]:
%sql
select my_upper('hello')

###SQL Query Files

In [0]:
%sql
create temporary view orders_csv
using csv
options (path "/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv",
header "true",
inferSchema "true")


In [0]:
%sql
select * from orders_csv;

In [0]:
%sql
create or replace temporary view orders_json
using json
options (path "/Volumes/sparkcatalog/raw/source/orders.json");

In [0]:
%sql
select * from orders_json;

###Spark SQL using DataFrame

In [0]:
df_csv=spark.read.format("csv")\
        .option("header",True)\
        .option("inferSchema",True)\
        .load("/Volumes/sparkcatalog/raw/source/raw_orders/orders.csv")

In [0]:
display(df_csv)

In [0]:
spark.sql("select * from {df_csv_var} where order_status='DELIVERED' ", df_csv_var=df_csv)

In [0]:
display(
    spark.sql('''
              select * from {df_csv_var}
              where price>10
              ''',
              df_csv_var=df_csv
              )
)

###Using File Format Connector

In [0]:
display(
spark.sql('''
          select * from json.`/Volumes/sparkcatalog/raw/source/orders.json`
          ''')
          )